# Phase 4A — Milestone 2: Label-Scheme Ablation Matrix

This notebook walks through and validates **Milestone 2** of Phase 4A: the
label-scheme ablation harness shipped in this branch.

**Reference documents**
- PRD: [`phase-4a-feature-and-label-redesign.prd.md`](../.claude/prds/phase-4a-feature-and-label-redesign.prd.md)
- Plan: [`phase-4a-milestone-2-label-ablation.plan.md`](../.claude/plans/phase-4a-milestone-2-label-ablation.plan.md)
- Concept doc: [`label-schemes.md`](../docs/concepts/label-schemes.md)
- Companion notebook: [`05_phase4a_regime_harness.ipynb`](05_phase4a_regime_harness.ipynb)

**Why this notebook exists.** Notebook 05 identified that the Phase 3 GBM's
failure attribution is dominated by `rate_cycle` (≈ −1.44 Sharpe vs ARIMA).
The dominant failure mode is **trend-fighting** — the model trains primarily
on high-vol crisis bars where mean-reversion pays, then mis-applies that
learning in the persistent uptrend of `rate_cycle`. This is a *label*
problem before it is a *feature* problem: signed forward return is scale-
ambiguous, has no PT/SL discipline, and is not path-dependent. Milestone 2
runs an ablation matrix over three label schemes:

| Scheme | What it fixes |
|---|---|
| `signed_returns` (control) | Phase 3 default — no fixes; baseline for the matrix. |
| `vol_scaled_returns` | Scale-ambiguity. Forward return ÷ σ̂[t] standardises the training signal across vol regimes. |
| `triple_barrier_labels` | PT/SL discipline + path-dependence (López de Prado AFML §3.5). |

The verdict pattern is **balanced multi-regime Borda count**: each column
in the per-regime Sharpe table (`aggregate`, `qe_bull`, `covid`,
`rate_cycle`) is ranked independently and the composite is the mean rank.
No regime is weighted more than another; the PRD success metric is about
multi-regime *breadth*, not aggregate Sharpe.

**Sections**
1. Setup and imports
2. Synthetic-data scheme demos
3. Pre-committed `LDP_DEFAULT` parameters
4. Real-data slice (5 symbols × ~8 years)
5. Run the ablation matrix
6. Per-regime Sharpe table (`ablation_summary_table`)
7. Balanced multi-regime Borda composite (`ablation_composite_ranking`)
8. Pairwise DM p-values per regime (`ablation_dm_matrix`)
9. Diagnostic appendix — focus on `rate_cycle`
10. Verdict + what's next

---
## 1 — Setup and imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

print(f'numpy   {np.__version__}')
print(f'pandas  {pd.__version__}')

In [ ]:
# Milestone 2 surface area — everything we will exercise in this notebook.
from quant.features.labels import LabelResult, generate_labels
from quant.features.label_schemes import (
    LDP_DEFAULT,
    TripleBarrierConfig,
    triple_barrier_labels,
    vol_scaled_returns,
)
from quant.backtest.ablation import run_label_ablation
from quant.backtest.regimes import DateRangeDetector, tag_regimes
from quant.backtest.report import (
    ablation_composite_ranking,
    ablation_dm_matrix,
    ablation_summary_table,
    format_ablation_report,
)
from quant.backtest.harness import BacktestResult, run_portfolio_backtest
from quant.backtest.metrics import compute_metrics
from quant.models.arima_baseline import ARIMABaseline

print('Milestone 2 surface area imported.')

---
## 2 — Synthetic-data scheme demos

Three handcrafted price paths exercise each scheme on a known signal so the
behaviour can be verified visually before moving to real data.

In [ ]:
def _synthetic_path(kind: str, n: int = 80, seed: int = 0) -> pd.Series:
    rng = np.random.default_rng(seed)
    if kind == 'ramp':
        log_returns = 0.004 + rng.normal(0, 0.002, n)
    elif kind == 'crash':
        log_returns = -0.004 + rng.normal(0, 0.002, n)
    elif kind == 'flat':
        log_returns = rng.normal(0, 0.005, n)
    else:
        raise ValueError(kind)
    levels = 100.0 * np.exp(np.cumsum(log_returns))
    dates = pd.bdate_range('2020-01-02', periods=n)
    return pd.Series(levels, index=dates, name=f'{kind}_price')


ramp = _synthetic_path('ramp', seed=1)
crash = _synthetic_path('crash', seed=2)
flat = _synthetic_path('flat', seed=3)

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, series, title in zip(axes, [ramp, crash, flat], ['ramp', 'crash', 'flat']):
    ax.plot(series.index, series.values, color='black', linewidth=0.9)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

### Scheme 1 — `signed_returns` (control)

Forward 1-bar return. Identical to the Phase 3 default label.

In [ ]:
signed = generate_labels(ramp, horizon=1)
print(f'signed_returns on ramp: horizon_bars={signed.horizon_bars}')
print(signed.series.dropna().describe()[['mean', 'std', 'min', 'max']].to_string())
print(f'sign counts: {np.sign(signed.series.dropna()).astype(int).value_counts().to_dict()}')

### Scheme 2 — `vol_scaled_returns`

Forward return ÷ trailing realised vol. Same numerator as `signed_returns`,
but the denominator standardises the training signal across vol regimes.

The `point-in-time` invariant says σ̂[t] is computed from returns ≤ t only.
Verify by checking that mutating future bars does not change σ̂[t].

In [ ]:
vol_scaled = vol_scaled_returns(ramp, horizon=1, vol_window=10)
print(f'vol_scaled on ramp: horizon_bars={vol_scaled.horizon_bars}')
print(vol_scaled.series.dropna().describe()[['mean', 'std', 'min', 'max']].to_string())

# Point-in-time check: mutate bars after t=30 → label at bar 20 must be unchanged.
ramp_mutated = ramp.copy()
ramp_mutated.iloc[30:] = ramp.iloc[30:].values * 5.0  # huge swings post-30
vol_scaled_mut = vol_scaled_returns(ramp_mutated, horizon=1, vol_window=10)

orig_at_20 = vol_scaled.series.iloc[20]
mut_at_20 = vol_scaled_mut.series.iloc[20]
print(f'\nvol_scaled[20] original: {orig_at_20:+.4f}')
print(f'vol_scaled[20] after future mutation: {mut_at_20:+.4f}')
# σ̂[20] uses returns 11..20 (point-in-time); the forward numerator uses
# bar 21, which we did NOT mutate. So bar 20's label is unchanged.
assert abs(orig_at_20 - mut_at_20) < 1e-10
print('✓ Point-in-time invariant holds for vol_scaled_returns at bar 20.')

### Scheme 3 — `triple_barrier_labels`

PT/SL barriers at ±σ̂ multiples, time-out at `max_horizon`. Path-dependent.

On the **ramp** path, PT should dominate; on **crash**, SL should dominate;
on **flat**, the labels should be distributed across {−1, 0, +1}.

In [ ]:
cfg = TripleBarrierConfig(pt_sigma=2.0, sl_sigma=1.0, vol_window=10, max_horizon=5)

for series, name in [(ramp, 'ramp'), (crash, 'crash'), (flat, 'flat')]:
    res = triple_barrier_labels(series, config=cfg)
    valid = res.series.dropna().astype(int)
    counts = valid.value_counts().to_dict()
    counts_str = ' '.join(f'{k:+d}:{counts.get(k, 0)}' for k in (-1, 0, 1))
    print(f'{name:<6}  horizon={res.horizon_bars}  n_valid={len(valid):3d}  {counts_str}')

---
## 3 — Pre-committed `LDP_DEFAULT` parameters

The triple-barrier defaults are pinned **before any model run**. Same
anti-p-hacking discipline as the VIX thresholds in `regime-evaluation.md`
and the T1–T6 gates in `evaluation-standards.md`. Override only via a PR
that re-runs the full ablation.

| Parameter | Default | Rationale |
|---|---|---|
| `pt_sigma` | 2.0 | AFML §3.5 asymmetric PT/SL — encodes "expect ≥ 2σ upside, survive 1σ adverse motion". |
| `sl_sigma` | 1.0 | 2:1 PT:SL paid for equity's long-run positive drift. |
| `vol_window` | 21 | One trading month — AFML §3.5; Bouchaud & Potters' "fast" vol horizon. |
| `max_horizon` | 5 | One trading week — AFML §3.5 sweet spot for daily bars. |

See [`docs/concepts/label-schemes.md`](../docs/concepts/label-schemes.md)
for the full citations and update protocol.

In [ ]:
print('LDP_DEFAULT:')
print(f'  pt_sigma    = {LDP_DEFAULT.pt_sigma}')
print(f'  sl_sigma    = {LDP_DEFAULT.sl_sigma}')
print(f'  vol_window  = {LDP_DEFAULT.vol_window}')
print(f'  max_horizon = {LDP_DEFAULT.max_horizon}')
assert LDP_DEFAULT == TripleBarrierConfig()  # config is frozen

---
## 4 — Real-data slice (5 symbols × ~8 years)

Mirrors nb05 §6 — a 5-symbol × ~8-year slice for fast iteration. The full-panel
ablation belongs to Milestone 6. The point here is to confirm the orchestrator
returns per-scheme `BacktestResult`s with populated `oos_returns` and
`oos_forecast_errors`, and that the reporter slices them by regime correctly.

In [ ]:
import quant.storage.catalog as catalog

DEMO_SYMBOLS = ['AAPL', 'MSFT', 'JPM', 'JNJ', 'SPY']
DEMO_START = '2018-01-02'
DEMO_END = '2026-04-21'

REAL_OHLCV_LOADED = False
prices_by_sym: dict[str, pd.DataFrame] = {}
try:
    syms_sql = ', '.join(f"'{s}'" for s in DEMO_SYMBOLS)
    eq = catalog.query(f"""
        SELECT symbol, timestamp, open, high, low, close, adjClose, volume
        FROM {catalog.table('equity_eod_tiingo')}
        WHERE symbol IN ({syms_sql})
          AND timestamp BETWEEN '{DEMO_START}' AND '{DEMO_END}'
        ORDER BY symbol, timestamp
    """)
    eq['timestamp'] = pd.to_datetime(eq['timestamp']).dt.tz_localize(None)
    eq = eq.set_index('timestamp')
    for s in DEMO_SYMBOLS:
        sdf = eq[eq['symbol'] == s][['open', 'high', 'low', 'close', 'volume']].copy()
        sdf = sdf[~sdf.index.duplicated(keep='last')].sort_index()
        if not sdf.empty:
            prices_by_sym[s] = sdf
    REAL_OHLCV_LOADED = bool(prices_by_sym)
    print(f'Loaded OHLCV for {len(prices_by_sym)} symbols')
    for s, df in prices_by_sym.items():
        print(f'  {s}: {len(df):,} bars  {df.index.min().date()} → {df.index.max().date()}')
except Exception as exc:
    print(f'Lake unavailable ({type(exc).__name__}): {exc}')

In [ ]:
# Trim each symbol so labels are defined on the same valid window across schemes.
# Triple-barrier requires the longest tail (max_horizon = 5), so trim by that.
TRIM_TAIL = LDP_DEFAULT.max_horizon
features_by_sym: dict[str, pd.DataFrame] = {}

if REAL_OHLCV_LOADED:
    for s, p in prices_by_sym.items():
        # A single placeholder column suffices for ARIMA (it ignores X).
        # Real GBM features would come from build_features but this notebook
        # is about the label side, not features.
        idx = p.index[:-TRIM_TAIL]
        features_by_sym[s] = pd.DataFrame({'placeholder': 0.0}, index=idx)
        prices_by_sym[s] = p.loc[idx]
    n_bars = sum(len(v) for v in features_by_sym.values())
    print(f'Built panel: {n_bars:,} (sym, bar) pairs across {len(features_by_sym)} symbols.')

---
## 5 — Run the ablation matrix

`run_label_ablation` iterates over the three schemes with identical
`train_window` / `test_window` / `step` / `embargo` parameters. The model
is `ARIMABaseline()` — a stable, fast control that lets us compare schemes
without the variance of a fresh GBM hyperparameter search. The Phase 4A
gate verdict (Milestone 6) will swap in `GBMModel`.

In [ ]:
SCHEMES = {
    'signed_returns': lambda p: generate_labels(p, horizon=1),
    'vol_scaled':     lambda p: vol_scaled_returns(p, horizon=1, vol_window=LDP_DEFAULT.vol_window),
    'triple_barrier': lambda p: triple_barrier_labels(p, config=LDP_DEFAULT),
}

ablation_results: dict[str, BacktestResult] = {}
if REAL_OHLCV_LOADED:
    ablation_results = run_label_ablation(
        label_schemes=SCHEMES,
        model=ARIMABaseline(),
        features_by_symbol=features_by_sym,
        prices_by_symbol=prices_by_sym,
        train_window=504,
        test_window=63,
        step=63,
        embargo=3,
    )
    print(f'Ran {len(ablation_results)} schemes through run_label_ablation.')
    for name, res in ablation_results.items():
        print(f'  {name:<18}  OOS bars={len(res.oos_returns):5d}  '
              f'Sharpe={res.oos_metrics["sharpe"]:+.3f}  '
              f'MaxDD={res.oos_metrics["max_drawdown"]:+.2%}')

---
## 6 — Per-regime Sharpe table

`ablation_summary_table` produces one row per scheme, columns = `aggregate`
plus each regime label. Cells are OOS Sharpe.

In [ ]:
era_labels = None
summary = None
if ablation_results:
    # All three schemes share fold structure (identical kwargs), so any
    # result's OOS index can be tagged once.
    any_result = next(iter(ablation_results.values()))
    era_labels = tag_regimes(any_result.oos_returns.index, DateRangeDetector())
    print('Regime composition of the OOS panel:')
    print(era_labels.value_counts().rename('bars').to_frame())

    summary = ablation_summary_table(ablation_results, era_labels)
    print('\nPer-regime OOS Sharpe by scheme:')
    display(summary.round(3))

---
## 7 — Balanced multi-regime Borda composite

`ablation_composite_ranking` ranks schemes 1 → N in each column (1 = best
Sharpe) and the composite is the mean rank across columns. No regime gets
special weighting. Lower composite is better.

In [ ]:
ranking = None
if ablation_results and era_labels is not None:
    ranking = ablation_composite_ranking(ablation_results, era_labels)
    print('Balanced multi-regime Borda composite ranking:')
    display(ranking.round(3))
    winner = ranking.index[0]
    print(f'\nComposite winner: {winner!r}  '
          f'(mean rank across regimes = {ranking.loc[winner, "mean_rank_across_regimes"]:.3f})')

---
## 8 — Pairwise DM p-values per regime

`ablation_dm_matrix` runs a Diebold-Mariano test (`alternative="less"`)
per scheme-pair per regime, on the forecast-error series. Cells are
p-values; smaller = stronger evidence the *row scheme* has smaller errors
than the *column scheme*.

Regimes with < 4 observations or degenerate variance return NaN rather
than crash.

In [ ]:
if ablation_results and era_labels is not None:
    dm_matrix = ablation_dm_matrix(ablation_results, era_labels)
    print('Pairwise DM p-values per regime (H1: A errors < B errors):')
    display(dm_matrix.round(4))

---
## 9 — Diagnostic appendix: focus on `rate_cycle`

The balanced Borda composite is the primary verdict. As a *secondary*
diagnostic — because nb05 flagged `rate_cycle` as the dominant failure
regime — we look at how each scheme behaves there specifically.

If the composite winner *also* wins `rate_cycle`, the scheme is a clear
advance. If the composite winner *loses* `rate_cycle`, the scheme is good
on average but does not address the original failure attribution, and
Milestone 3 (regime-aware features) becomes more important.

In [ ]:
if ablation_results and summary is not None and ranking is not None:
    if 'rate_cycle' not in summary.columns:
        print('rate_cycle regime not present in this slice (panel too short?).')
    else:
        rate_cycle_sharpes = summary['rate_cycle'].sort_values(ascending=False)
        print('rate_cycle OOS Sharpe by scheme (descending):')
        for scheme, sharpe in rate_cycle_sharpes.items():
            comp_rank = int(ranking.loc[scheme, 'composite_rank'])
            print(f'  {scheme:<18}  Sharpe={sharpe:+.3f}  composite_rank={comp_rank}')

        composite_winner = ranking.index[0]
        rate_cycle_winner = rate_cycle_sharpes.index[0]
        if composite_winner == rate_cycle_winner:
            print(f'\n→ Composite winner {composite_winner!r} also wins rate_cycle.')
            print('  Scheme advances; Milestone 3 features can build on it.')
        else:
            print(f'\n→ Composite winner {composite_winner!r} but rate_cycle winner {rate_cycle_winner!r}.')
            print('  No scheme alone fixes the rate_cycle failure regime.')
            print('  Milestone 3 (regime-aware features) becomes the next bet.')

---
## 10 — Verdict + what's next

The ablation matrix above is computed on a **5-symbol × ~8-year** slice,
not the full 33-symbol × 23-year Phase 4A panel. Milestone 6 will re-run
the same harness on the full panel and ARIMA-vs-GBM (not ARIMA control)
to determine the Phase 4A gate verdict.

**Hand-off to next milestone.**

- **If the slice winner is the control** (`signed_returns`), no scheme
  advances → Milestone 3 (regime-aware features) carries the work forward.
- **If a non-control scheme wins** (`vol_scaled` or `triple_barrier`),
  promote it to the M3 baseline label — Milestone 3 features layer on top.
- **If the composite winner also wins `rate_cycle`**, the scheme directly
  addresses nb05's failure attribution. That's the strongest outcome.

**Triple-barrier deferral candidate (Milestone 2.5).**

The PRD reserves a sub-milestone for **meta-labeling on the M2 winner**
(López de Prado AFML §3.6). The trigger is: a non-control scheme beats
ARIMA in at least one regime. If that trigger fires, M2.5 builds a
two-stage primary-then-meta harness on the winning scheme. If not (no
edge from any scheme), M2.5 is skipped — meta-labeling cannot rescue a
primary with no edge — and the project goes straight to M3.

See:
- PRD § Milestone 2.5 trigger criteria
- [`docs/concepts/label-schemes.md`](../docs/concepts/label-schemes.md)

In [ ]:
if ablation_results and era_labels is not None:
    print('=' * 70)
    print('FULL FORMATTED ABLATION REPORT')
    print('=' * 70)
    print(format_ablation_report(ablation_results, era_labels))